# 08 - Publication Figures & Visual Storyboard Pipeline

This notebook generates the **5+1 Curated Thesis Figure Suite** at publication-grade 300 DPI (`scale=2`). Every figure cell is paired with an explicit **Thesis Storyboard Card** answering:
1. 🎯 **Research Question Answered**
2. 💡 **Scientific Motivation ("Why are we plotting this data?")**
3. 🔍 **Visual Interpretation Guide (Axes, Baselines, Significance)**
4. 📝 **Key Findings for Thesis Drafting**

---

### 🖼️ The 5+1 Curated Figure Suite:
- **Figure A:** Problem Convergence & Precision Dashboard (4-Panel)
- **Figure B:** Clean-to-Noisy Matched-Pair Generalizability Transfer Scatter
- **Figure C:** Noise Degradation & Landscape Fragility Index Matrix
- **Figure D:** Dolan-Moré Empirical Performance Profiles $\rho_s(\tau)$
- **Figure E:** Pairwise Vargha-Delaney ($A_{12}$) Stochastic Dominance Heatmap
- **Figure F (Appendix):** Per-Problem Convergence Curves & Target ECDF Trajectories (All Solvers in 1 Figure)


In [19]:
# ── 1. Setup Environment, Paths & Publication Theme ───────────────────────────
import os
import re
import sys
import json
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, pearsonr

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, RESULTS_DIR

DB_PATH      = DATA_DIR / 'db.sqlite3'
IOH_LOGS_DIR = DATA_DIR / 'ioh_logs'
FIGURES_DIR  = RESULTS_DIR / 'figures'
ADVANCED_DIR = FIGURES_DIR / 'advanced'
ADVANCED_DIR.mkdir(parents=True, exist_ok=True)

BBOB_NAMES = {1: 'Sphere (f1)', 8: 'Rosenbrock (f8)', 11: 'Discus (f11)', 15: 'Rastrigin (f15)', 21: 'Gallagher 101 Peaks (f21)'}
BBOB_CLASSES = {1: 'Separable', 8: 'Low Conditioning', 11: 'High Conditioning', 15: 'Multi-Modal (Global)', 21: 'Multi-Modal (Weak)'}

SOLVER_COLORS = {
    'CMAES': '#636EFA', 'DE': '#EF553B', 'PSO': '#00CC96',
    'LLaMEA_Baseline': '#AB63FA', 'LLaMEA_Thinking': '#FFA15A',
    'LLaMEA_Vectorization': '#19D3F3', 'LLaMEA_Guided': '#FF6692', 'LLaMEA_Champion': '#FFD700',
    'LLaMEA_Evolved': '#9C27B0'
}

def apply_publication_theme(fig, title=None, width=950, height=540, top_margin=95):
    fig.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>{title}</b>' if title else None,
            x=0.03,
            y=0.98,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#333333'),
        margin=dict(l=65, r=45, t=top_margin if title else 40, b=55),
        width=width, height=height,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5,
            bgcolor='rgba(255,255,255,0.92)',
            bordercolor='rgba(0,0,0,0.12)',
            borderwidth=1,
            font=dict(size=11)
        )
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False)
    return fig

print('✅ Figure environment initialized (PNG-only export, 300 DPI).')


✅ Figure environment initialized (PNG-only export, 300 DPI).


In [20]:
# ── 2. Data Parsers for IOH Logs & SQLite Database ───────────────────────────
def parse_ioh_dat_file(dat_path: Path):
    runs = []
    current_evals, current_raw = [], []
    with open(dat_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith(('function', 'evaluations', '"evaluations"', '#', 'instance')):
                if current_evals: runs.append((np.array(current_evals), np.array(current_raw))); current_evals, current_raw = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                try: current_evals.append(float(parts[0])); current_raw.append(float(parts[1]))
                except ValueError: continue
    if current_evals: runs.append((np.array(current_evals), np.array(current_raw)))
    return runs

def load_benchmark_ioh_data(ioh_dir: Path):
    data_store = {}
    if not ioh_dir.exists(): return data_store
    for json_path in ioh_dir.glob('**/*.json'):
        try:
            with open(json_path, 'r') as jf: meta = json.load(jf)
        except Exception: continue
        path_str = str(json_path.relative_to(ioh_dir))
        dim_m = re.search(r'(\d+)D', path_str); dim = int(dim_m.group(1)) if dim_m else None
        noise_m = re.search(r'std_([\d\.]+)', path_str); noise_std = float(noise_m.group(1)) if noise_m else 0.0
        p_id = meta.get('function_id')
        if p_id is None: p_m = re.search(r'f(\d+)', path_str); p_id = int(p_m.group(1)) if p_m else None
        parent_name = json_path.parent.name
        if 'dummy' in parent_name.lower(): continue
        solver_name = next((c for c in ['CMAES', 'DE', 'PSO', 'LLaMEA_Baseline', 'LLaMEA_Thinking', 'LLaMEA_Vectorization', 'LLaMEA_Guided', 'LLaMEA_Champion'] if c.lower() in parent_name.lower()), 'LLaMEA_Evolved' if 'llamea' in parent_name.lower() else parent_name.split('_')[0])
        for sc in meta.get('scenarios', []):
            if dim is None: dim = sc.get('dimension')
            if p_id is None or dim is None: continue
            key = (dim, noise_std, p_id)
            if key not in data_store: data_store[key] = {}
            if solver_name not in data_store[key]: data_store[key][solver_name] = []
            dat_p = sc.get('path')
            if dat_p and (json_path.parent / dat_p).exists(): data_store[key][solver_name].extend(parse_ioh_dat_file(json_path.parent / dat_p))
    return data_store

def load_sqlite_synthesis_data(db_path: Path):
    if not db_path.exists(): return pd.DataFrame(), pd.DataFrame()
    conn = sqlite3.connect(db_path)
    df_exp = pd.read_sql_query('SELECT * FROM experiments', conn)
    df_iter = pd.read_sql_query('SELECT i.id AS iteration_id, i.experiment_id, i.algorithm_name, i.raw_fitness, i.final_error, i.timed_out, i.converged, i.runtime_seconds, e.problem_id, e.dim, e.mode, e.llm_name, e.prompt_strategy FROM iterations i JOIN experiments e ON i.experiment_id = e.id', conn)
    conn.close()
    return df_exp, df_iter

print('✅ Parsers loaded successfully.')


✅ Parsers loaded successfully.


In [21]:
# ── 3. Load Datasets from SQLite and IOH Logs ─────────────────────────────────
df_exp, df_iter = load_sqlite_synthesis_data(DB_PATH)
all_benchmark_data = load_benchmark_ioh_data(IOH_LOGS_DIR)
all_solvers = sorted(list(set(s for cond in all_benchmark_data.values() for s in cond.keys())))
print(f'📦 Ready to render figures across {len(all_benchmark_data)} benchmark problem conditions.')


📦 Ready to render figures across 20 benchmark problem conditions.


## 📊 Figure A: Problem Convergence & Precision Dashboard (4-Panel)

- 🎯 **Research Question Answered:** Can LLaMEA-discovered algorithms converge across diverse continuous landscape classes (Separable, Conditioning, Multimodal), and what precision order of magnitude is achieved?
- 💡 **Why We Plot This Data:** Establishes the foundational feasibility of LLM-generated code across the BBOB testbed. It highlights the contrast between easily solvable landscapes ($f_1$ Sphere, $f_{11}$ Discus reaching machine precision) versus structural valley traps ($f_8$ Rosenbrock stagnation).
- 🔍 **Visual Guide:**
  - **Panel (A):** Success rate (%) under Clean ($\sigma=0$) vs. Noisy ($\sigma=0.05$) conditions.
  - **Panel (B):** Median precision achieved ($-\log_{10}(\Delta y)$) for successful runs.
  - **Panel (C):** Status breakdown: High Precision ($<10^{-5}$), Moderate, Stagnated, or Non-Converged ($>10^8$).
  - **Panel (D):** Resilience per solver on the hardest landscape: Rosenbrock ($f_8$).


In [22]:
# ── Figure A: Problem Convergence & Precision Dashboard ───────────────────────
prob_stats = []
for (dim, noise_std, p_id), solvers_dict in all_benchmark_data.items():
    p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
    p_class = BBOB_CLASSES.get(p_id, 'Unknown')
    mode_lbl = 'Noisy (σ=0.05)' if noise_std > 0 else 'Clean (σ=0.0)'
    for solver, runs in solvers_dict.items():
        terminals = [r[1][-1] for r in runs if len(r[1]) > 0]
        for t in terminals:
            is_non_conv = (t >= 1e8 or np.isnan(t))
            precision = 0.0 if is_non_conv else -np.log10(max(1e-12, t))
            status = 'Non-Converged' if is_non_conv else ('High Precision' if t <= 1e-5 else ('Moderate' if t <= 1e-2 else 'Stagnated'))
            prob_stats.append({'dim': dim, 'noise_std': noise_std, 'mode': mode_lbl, 'problem_id': p_id, 'problem_name': p_name, 'class': p_class, 'solver': solver, 'terminal_error': t, 'precision': precision, 'is_converged': not is_non_conv, 'status': status})

df_pconv = pd.DataFrame(prob_stats)
if not df_pconv.empty:
    fig_prob_conv = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            '<b>(A) Convergence Success Rate by Problem</b>',
            '<b>(B) Median Precision Achieved: -log10(Δy)</b>',
            '<b>(C) Operational Status Breakdown (%)</b>',
            '<b>(D) Solver Resilience on Hardest Landscape: Rosenbrock (f8)</b>'
        ),
        vertical_spacing=0.25,
        horizontal_spacing=0.12
    )
    # Panel A
    agg_succ = df_pconv.groupby(['problem_name', 'mode'])['is_converged'].mean().reset_index(); agg_succ['pct'] = agg_succ['is_converged'] * 100
    for m_lbl, col in [('Clean (σ=0.0)', '#2ecc71'), ('Noisy (σ=0.05)', '#e74c3c')]:
        sub = agg_succ[agg_succ['mode'] == m_lbl]
        fig_prob_conv.add_trace(go.Bar(x=sub['problem_name'], y=sub['pct'], name=m_lbl, marker_color=col, showlegend=True), row=1, col=1)
    # Panel B
    agg_prec = df_pconv[df_pconv['is_converged']].groupby(['problem_name', 'mode'])['precision'].median().reset_index()
    for m_lbl, col in [('Clean (σ=0.0)', '#2ecc71'), ('Noisy (σ=0.05)', '#e74c3c')]:
        sub = agg_prec[agg_prec['mode'] == m_lbl]
        fig_prob_conv.add_trace(go.Bar(x=sub['problem_name'], y=sub['precision'], name=m_lbl, marker_color=col, showlegend=False), row=1, col=2)
    # Panel C
    status_order = ['High Precision', 'Moderate', 'Stagnated', 'Non-Converged']
    status_colors = {'High Precision': '#27ae60', 'Moderate': '#2980b9', 'Stagnated': '#f39c12', 'Non-Converged': '#c0392b'}
    agg_stat = df_pconv.groupby(['problem_name', 'status']).size().unstack(fill_value=0); agg_stat_pct = agg_stat.div(agg_stat.sum(axis=1), axis=0) * 100
    for st in status_order:
        if st in agg_stat_pct.columns: fig_prob_conv.add_trace(go.Bar(x=agg_stat_pct.index, y=agg_stat_pct[st], name=st, marker_color=status_colors[st], showlegend=True), row=2, col=1)
    # Panel D
    df_f8 = df_pconv[df_pconv['problem_id'] == 8]
    if not df_f8.empty:
        agg_f8 = df_f8.groupby(['solver', 'mode'])['is_converged'].mean().reset_index(); agg_f8['pct'] = agg_f8['is_converged'] * 100
        for m_lbl, col in [('Clean (σ=0.0)', '#2ecc71'), ('Noisy (σ=0.05)', '#e74c3c')]:
            sub = agg_f8[agg_f8['mode'] == m_lbl]
            fig_prob_conv.add_trace(go.Bar(x=sub['solver'], y=sub['pct'], name=m_lbl, marker_color=col, showlegend=False), row=2, col=2)
    fig_prob_conv.update_layout(
        barmode='group', template='plotly_white',
        width=1150, height=800,
        margin=dict(l=60, r=40, t=100, b=65),
        legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='center', x=0.5, bgcolor='rgba(255,255,255,0.92)', bordercolor='rgba(0,0,0,0.12)', borderwidth=1)
    )
    fig_prob_conv.update_yaxes(title='<b>Success Rate (%)</b>', range=[0, 115], row=1, col=1)
    fig_prob_conv.update_yaxes(title='<b>Precision: -log10(Δy)</b>', row=1, col=2)
    fig_prob_conv.update_yaxes(title='<b>% of Total Runs</b>', range=[0, 105], row=2, col=1)
    fig_prob_conv.update_yaxes(title='<b>f8 Convergence (%)</b>', range=[0, 115], row=2, col=2)
    fig_prob_conv.update_xaxes(tickangle=-15, row=1, col=1)
    fig_prob_conv.update_xaxes(tickangle=-15, row=1, col=2)
    fig_prob_conv.update_xaxes(tickangle=-15, row=2, col=1)
    fig_prob_conv.update_xaxes(tickangle=-15, row=2, col=2)
    out_p = ADVANCED_DIR / 'problem_convergence_comparison.png'
    fig_prob_conv.write_image(str(out_p), scale=2)
    print(f'  ✅ Figure A Exported -> {out_p.name}')


  ✅ Figure A Exported -> problem_convergence_comparison.png


## 🔬 Figure B: Clean-to-Noisy Matched-Pair Generalizability Transfer

- 🎯 **Research Question Answered:** Does algorithm optimization performance achieved under deterministic (Clean) synthesis predict performance when deployed in stochastic (Noisy) environments?
- 💡 **Why We Plot This Data:** Connects the **synthesis database** (`data/db.sqlite3`) with downstream robustness. A strong correlation along the diagonal ($y=x$) proves that algorithms evolved on clean functions generalize well to noisy functions, rather than overfitting to zero-noise artifacts.
- 🔍 **Visual Guide:** Each point represents a matched experimental trial $(D, \text{Strategy}, \text{Model})$. The dashed line is perfect transfer ($y = x$). Points above the diagonal indicate performance degradation under noise.


In [23]:
# ── Figure B: Clean-to-Noisy Matched-Pair Transfer ────────────────────────────
if not df_exp.empty:
    df_clean = df_exp[df_exp['mode'].astype(str).str.lower() == 'clean'].copy()
    df_noisy = df_exp[df_exp['mode'].astype(str).str.lower() == 'noisy'].copy()
    match_keys = ['problem_id', 'dim', 'prompt_strategy', 'llm_name']
    df_matched = pd.merge(df_clean[match_keys + ['best_final_error']], df_noisy[match_keys + ['best_final_error']], on=match_keys, suffixes=('_clean', '_noisy'))
    df_matched['log_clean'] = np.log10(np.clip(df_matched['best_final_error_clean'].astype(float), 1e-12, 1e9))
    df_matched['log_noisy'] = np.log10(np.clip(df_matched['best_final_error_noisy'].astype(float), 1e-12, 1e9))
    valid = df_matched[(df_matched['best_final_error_clean'] < 1e8) & (df_matched['best_final_error_noisy'] < 1e8)]
    r_val, p_val = (pearsonr(valid['log_clean'], valid['log_noisy']) if len(valid) >= 3 else (0.0, 1.0))
    fig_transfer = go.Figure()
    for p_id in sorted(df_matched['problem_id'].unique()):
        sub = df_matched[df_matched['problem_id'] == p_id]
        fig_transfer.add_trace(go.Scatter(x=sub['log_clean'], y=sub['log_noisy'], mode='markers', name=BBOB_NAMES.get(p_id, f'f{p_id}'), marker=dict(size=11, opacity=0.85, line=dict(width=1, color='#2c3e50')), text=[f"{r['prompt_strategy']} ({r['dim']}D)" for _, r in sub.iterrows()]))
    diag_range = [-12, 9]
    fig_transfer.add_trace(go.Scatter(x=diag_range, y=diag_range, mode='lines', line=dict(color='#888888', dash='dash', width=1.5), name='Perfect Transfer (y = x)', hoverinfo='skip'))
    fig_transfer.update_xaxes(title='<b>Clean Mode Error</b> [log10(Δy)]', range=[-13, 10])
    fig_transfer.update_yaxes(title='<b>Noisy Mode Error</b> [log10(Δy)]', range=[-13, 10])
    apply_publication_theme(fig_transfer, title=f'Figure B: Clean-to-Noisy Generalizability Transfer (r = {r_val:.2f}, p = {p_val:.2e})', width=920, height=540, top_margin=95)
    out_p = ADVANCED_DIR / 'clean_vs_noisy_transfer.png'
    fig_transfer.write_image(str(out_p), scale=2)
    print(f'  ✅ Figure B Exported -> {out_p.name} (r={r_val:.2f}, p={p_val:.2e})')


  ✅ Figure B Exported -> clean_vs_noisy_transfer.png (r=0.32, p=1.14e-01)


## 🌊 Figure C: Noise Degradation & Landscape Fragility Index Matrix

- 🎯 **Research Question Answered:** Which problem landscapes and algorithm architectures suffer the greatest performance degradation when subjected to stochastic evaluation noise ($\sigma = 0.05$)?
- 💡 **Why We Plot This Data:** Identifies structural vulnerabilities. Classical optimizers rely on precise gradient approximations that break in stochastic valleys ($f_8$), whereas evolutionary LLM algorithms show different sensitivity profiles.
- 🔍 **Visual Guide:** Cells display the Degradation Index $\Delta \log_{10}(\Delta y) = \log_{10}(\text{Median Error}_{\text{Noisy}}) - \log_{10}(\text{Median Error}_{\text{Clean}})$. Warm colors (purple/orange) represent severe precision loss, while zero represents perfect noise resilience.


In [24]:
# ── Figure C: Noise Degradation Matrix ─────────────────────────────────────────
prob_ids = sorted(list(set(k[2] for k in all_benchmark_data.keys())))
deg_grid = np.zeros((len(prob_ids), len(all_solvers)))
deg_text = []
for i, p_id in enumerate(prob_ids):
    row_t = []
    for j, solver in enumerate(all_solvers):
        c_runs, n_runs = [], []
        for (dim, n_std, pid), s_dict in all_benchmark_data.items():
            if pid == p_id and solver in s_dict:
                terms = [r[1][-1] for r in s_dict[solver] if len(r[1]) > 0]
                if n_std == 0.0: c_runs.extend(terms)
                else: n_runs.extend(terms)
        if c_runs and n_runs:
            c_med, n_med = np.median(c_runs), np.median(n_runs)
            deg = np.log10(max(1e-12, n_med)) - np.log10(max(1e-12, c_med))
            deg_grid[i, j] = deg; row_t.append(f'{deg:+.1f}')
        else: deg_grid[i, j] = 0.0; row_t.append('N/A')
    deg_text.append(row_t)
fig_deg = go.Figure(data=go.Heatmap(z=deg_grid, x=all_solvers, y=[BBOB_NAMES.get(p, f'f{p}') for p in prob_ids], text=deg_text, texttemplate='%{text}', textfont=dict(size=11), colorscale='Plasma', zmid=0.0, colorbar=dict(title='<b>Degradation</b><br>Δlog10(Error)')))
apply_publication_theme(fig_deg, title='Figure C: Noise Degradation Factor Matrix across Problem Landscapes', width=920, height=480, top_margin=90)
fig_deg.update_xaxes(tickangle=-20)
out_p = ADVANCED_DIR / 'noise_degradation_matrix.png'
fig_deg.write_image(str(out_p), scale=2)
print(f'  ✅ Figure C Exported -> {out_p.name}')


  ✅ Figure C Exported -> noise_degradation_matrix.png


## 📈 Figure D: Dolan-Moré Empirical Performance Profiles $\rho_s(\tau)$

- 🎯 **Research Question Answered:** How does LLaMEA rank in overall benchmark coverage and efficiency compared to classical baselines (CMA-ES, DE, PSO)?
- 💡 **Why We Plot This Data:** **Gold standard in continuous optimization literature (BBOB / CEC).** Avoids the bias of arithmetic averaging across heterogeneous problem scales. $\tau=1$ measures the fraction of problems where a solver is the single fastest/best (zero-slack), while $\tau \gg 1$ reflects global robustness.
- 🔍 **Visual Guide:** Higher curves dominate. At $\tau = 1$, the solver with the highest intercept is the top-performing algorithm. As $\tau \to \infty$, the curve indicates the asymptotic problem solve rate.


In [25]:
# ── Figure D: Dolan-Moré Performance Profiles ──────────────────────────────────
prob_keys = list(all_benchmark_data.keys())
perf_matrix = {s: {} for s in all_solvers}
for p_key in prob_keys:
    for s in all_solvers:
        terms = [r[1][-1] for r in all_benchmark_data[p_key].get(s, []) if len(r[1]) > 0]
        perf_matrix[s][p_key] = np.median(terms) if terms else 1e9
best_perf = {p_key: min([perf_matrix[s][p_key] for s in all_solvers]) + 1e-12 for p_key in prob_keys}
tau_grid = np.logspace(0, 4, 150)
dolan_curves = {s: [] for s in all_solvers}
for tau in tau_grid:
    for s in all_solvers:
        solved = sum(1 for p_key in prob_keys if (perf_matrix[s][p_key] + 1e-12) / best_perf[p_key] <= tau)
        dolan_curves[s].append(solved / max(1, len(prob_keys)))
fig_dolan = go.Figure()
for s in all_solvers:
    fig_dolan.add_trace(go.Scatter(x=tau_grid, y=dolan_curves[s], mode='lines', name=s, line=dict(color=SOLVER_COLORS.get(s, '#7f7f7f'), width=2.5 if 'LLaMEA' in s else 1.5, dash='solid' if 'LLaMEA' in s else 'dash')))
fig_dolan.update_xaxes(type='log', title='<b>Performance Ratio Factor (τ)</b>')
fig_dolan.update_yaxes(title='<b>Fraction of Problems Solved (ρ(τ))</b>', range=[-0.02, 1.02])
apply_publication_theme(fig_dolan, title='Figure D: Dolan-Moré Performance Profiles ρ(τ) across all Benchmark Problems', width=920, height=540, top_margin=95)
out_p = ADVANCED_DIR / 'dolan_more_profiles.png'
fig_dolan.write_image(str(out_p), scale=2)
print(f'  ✅ Figure D Exported -> {out_p.name}')


  ✅ Figure D Exported -> dolan_more_profiles.png


## 🗺️ Figure E: Pairwise Vargha-Delaney ($A_{12}$) Effect Size Heatmap

- 🎯 **Research Question Answered:** Are performance differences between LLaMEA variants and classical baselines statistically significant and of meaningful practical magnitude?
- 💡 **Why We Plot This Data:** Replaces simple p-values with **effect sizes**. $A_{12} > 0.5$ represents the probability that the Row algorithm achieves a better (lower) objective value than the Column algorithm. Asterisks indicate two-sided significance ($p < 0.05^*, p < 0.01^{**}, p < 0.001^{***}$).
- 🔍 **Visual Guide:** Green cells ($A_{12} > 0.5$) denote Row dominance; Red cells denote Column dominance; Neutral (0.5) denotes parity.


In [26]:
# ── Figure E: Pairwise A12 Effect Size Heatmap ────────────────────────────────
def vargha_delaney_a12(sample1, sample2):
    m, n = len(sample1), len(sample2)
    if m == 0 or n == 0: return 0.5, 'negligible'
    r1 = np.sum([np.sum(x < sample2) + 0.5 * np.sum(x == sample2) for x in sample1])
    a12 = r1 / (m * n)
    return float(a12)

solver_residuals = {s: [] for s in all_solvers}
for p_key, s_dict in all_benchmark_data.items():
    for s in all_solvers:
        terms = [r[1][-1] for r in s_dict.get(s, []) if len(r[1]) > 0]
        solver_residuals[s].extend(terms)

a12_grid = np.zeros((len(all_solvers), len(all_solvers)))
text_grid = []
for i, s1 in enumerate(all_solvers):
    row_text = []
    for j, s2 in enumerate(all_solvers):
        if i == j: a12_grid[i, j] = 0.5; row_text.append('—')
        else:
            v1, v2 = solver_residuals[s1], solver_residuals[s2]
            if v1 and v2:
                a12 = vargha_delaney_a12(v1, v2); a12_grid[i, j] = a12
                try:
                    p_val = mannwhitneyu(v1, v2, alternative='two-sided').pvalue
                    ast = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
                except Exception: ast = ''
                row_text.append(f'{a12:.2f} {ast}')
            else: a12_grid[i, j] = 0.5; row_text.append('N/A')
    text_grid.append(row_text)

fig_a12 = go.Figure(data=go.Heatmap(z=a12_grid, x=all_solvers, y=all_solvers, text=text_grid, texttemplate='%{text}', textfont=dict(size=11), colorscale='RdYlGn', zmin=0.0, zmax=1.0, colorbar=dict(title='<b>A12 (Row < Col)</b><br>Green = Row Wins')))
apply_publication_theme(fig_a12, title='Figure E: Global Pairwise Effect Size Matrix (Vargha-Delaney A12)', width=880, height=620, top_margin=90)
fig_a12.update_xaxes(tickangle=-20)
out_p = ADVANCED_DIR / 'a12_effect_size_heatmap.png'
fig_a12.write_image(str(out_p), scale=2)
print(f'  ✅ Figure E Exported -> {out_p.name}')


  ✅ Figure E Exported -> a12_effect_size_heatmap.png


## 📑 Figure F (Appendix): Per-Problem Convergence Curves & Target ECDFs

- 🎯 **Research Question Answered:** What does the exact empirical iteration-by-iteration optimization trajectory look like for each individual test function?
- 💡 **Why We Plot This Data:** Supplementary thesis material. Provides full transparency into search dynamics: median error decay curves $\pm 1\sigma$ alongside the empirical cumulative distribution of target hits ($10^{-8} \dots 10^2$). **Unifies all classical and evolved algorithms into a single 2-panel chart without split folders.**
- 🔍 **Visual Guide:**
  - **Panel 1 (Left):** Log-log plot of evaluations vs. best objective value.
  - **Panel 2 (Right):** Target precision (reversed log scale) vs. fraction of runs reaching the target.


In [27]:
# ── Figure F: Per-Problem Convergence & ECDF Curves (Appendix) ─────────────────
eval_grid = np.logspace(0, 5, 200)
targets = np.logspace(-8, 2, 100)
exported_appendix_figures = 0

conditions_set = sorted(list(set((k[0], k[1]) for k in all_benchmark_data.keys())))
for (dim, noise_std) in conditions_set:
    cond_dir = FIGURES_DIR / f'{dim}D' / f'std_{noise_std}'
    cond_dir.mkdir(parents=True, exist_ok=True)
    for p_id in prob_ids:
        key = (dim, noise_std, p_id)
        if key not in all_benchmark_data: continue
        algo_runs = all_benchmark_data[key]
        p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
        p_class = BBOB_CLASSES.get(p_id, 'Unknown')
        
        fig_p = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                '<b>(A) Empirical Convergence Trajectory</b>',
                '<b>(B) Target Precision Hit Rate (ECDF)</b>'
            ),
            horizontal_spacing=0.14
        )
        for solver, runs in algo_runs.items():
            if not runs: continue
            col = SOLVER_COLORS.get(solver, '#7f7f7f')
            interpolated = [np.interp(eval_grid, evals, raw_vals, left=raw_vals[0], right=raw_vals[-1]) for evals, raw_vals in runs if len(evals) > 0]
            if interpolated:
                fig_p.add_trace(go.Scatter(x=eval_grid, y=np.median(np.array(interpolated), axis=0), mode='lines', name=solver, line=dict(color=col, width=2.2)), row=1, col=1)
            terminals = [r[1][-1] for r in runs if len(r[1]) > 0]
            if terminals:
                fig_p.add_trace(go.Scatter(x=targets, y=[np.mean(np.array(terminals) <= t) for t in targets], mode='lines', name=solver, line=dict(color=col, width=2.2), showlegend=False), row=1, col=2)
                
        super_title = f'BBOB f{p_id}: {p_name} — {dim}D (Noise: σ = {noise_std}, Class: {p_class})'
        fig_p.update_layout(
            template='plotly_white',
            title=dict(
                text=f'<b>{super_title}</b>',
                x=0.03,
                y=0.98,
                font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
            ),
            margin=dict(l=65, r=40, t=125, b=60),
            width=1000, height=480,
            legend=dict(
                orientation='h',
                yanchor='bottom',
                y=1.10,
                xanchor='center',
                x=0.5,
                bgcolor='rgba(255,255,255,0.92)',
                bordercolor='rgba(0,0,0,0.12)',
                borderwidth=1,
                font=dict(size=11)
            )
        )
        fig_p.update_xaxes(type='log', title='<b>Evaluations</b>', showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=1)
        fig_p.update_yaxes(type='log', title='<b>Best Fitness Value (Δy)</b>', showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=1)
        fig_p.update_xaxes(type='log', title='<b>Target Precision (τ)</b>', autorange='reversed', showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=2)
        fig_p.update_yaxes(title='<b>Fraction of Runs Solved</b>', range=[-0.05, 1.05], showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=2)
        
        out_p_png = cond_dir / f'f{p_id}_all_solvers.png'
        fig_p.write_image(str(out_p_png), scale=2)
        exported_appendix_figures += 1

print(f'  ✅ Exported {exported_appendix_figures} per-problem appendix figures (Figure F)')


  ✅ Exported 20 per-problem appendix figures (Figure F)
